In [11]:
import sqlite3

import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

import numpy as np

import cv2


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



/home/leonard/.local/lib/python3.10/site-packages/IPython/core/completerlib.py:150: UserWarning: This is now an optional IPython functionality, setting rootmodules_cache requires you to install the `pickleshare` library.
  ip.db['rootmodules_cache'] = rootmodules_cache


In [5]:
db_path = "/mnt/d/test/test_0.db3"

with sqlite3.connect(db_path) as conn:
	cur = conn.cursor()
	
	cur.execute("SELECT id, name, type FROM topics")
	for tid, name, typ in cur.fetchall():
		print(tid, name, typ)

# con.close()

1 /camera/camera/color/image_raw/compressed sensor_msgs/msg/CompressedImage
2 /camera/camera/color/image_raw sensor_msgs/msg/Image
3 /camera/camera/color/camera_info sensor_msgs/msg/CameraInfo
4 /camera/camera/aligned_depth_to_color/image_raw sensor_msgs/msg/Image
5 /camera/camera/aligned_depth_to_color/camera_info sensor_msgs/msg/CameraInfo


In [20]:

with sqlite3.connect(db_path) as conn:
	cur = conn.cursor()
	cur.execute("""
		SELECT m.timestamp, m.data
		FROM messages m
		JOIN topics t ON m.topic_id = t.id
		WHERE t.name = ?
		ORDER BY m.timestamp
	""", ("/camera/camera/color/camera_info",))

	data = cur.fetchall()
	print(data)

[(1769674926645789884, b'\x00\x01\x00\x00\xae\x18{i\x05\x16\x0c#\x1b\x00\x00\x00camera_color_optical_frame\x00\x00\xe0\x01\x00\x00\x80\x02\x00\x00\n\x00\x00\x00plumb_bob\x00\x00\x00\x05\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xe0u\xec\x82@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00 \xfa\x06t@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00@\x8b\xec\x82@\x00\x00\x00@\x97&o@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xf0?\x00\x00\x00\x00\x00\x00\xf0?\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xf0?\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xf0?\x00\x00\x00\xe0u\xec\x82@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00 \xfa\x06t@\x00\x00\x00\x0

In [ ]:
topic_name = "/camera/camera/color/image_raw/compressed"

writer = None
with sqlite3.connect(db_path) as conn:
	cur = conn.cursor()
	cur.execute("""
		SELECT m.timestamp, m.data
		FROM messages m
		JOIN topics t ON m.topic_id = t.id
		WHERE t.name = ?
		ORDER BY m.timestamp
	""", (topic_name,))

	for timestamp, data in cur:   # row-by-row fetch
		msg = deserialize_message(data, get_message("sensor_msgs/msg/CompressedImage"))
		img_np = np.frombuffer(msg.data, dtype=np.uint8)
		frame = cv2.imdecode(img_np, cv2.IMREAD_COLOR)

		
		if writer is None:
			h, w, _ = frame.shape
			fourcc = cv2.VideoWriter_fourcc(*"mp4v")
			writer = cv2.VideoWriter("./video.mp4", fourcc, 30, (w, h))
			
		# break
		writer.write(frame)

if writer is not None:
    writer.release()

In [14]:
img_np = np.frombuffer(msg.data, dtype=np.uint8)
img_np = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
img_np = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

cv2.imshow('Image', img_np)
cv2.waitKey(0)
cv2.destroyAllWindows()